# Muon Four-Momentum Reconstruction

In this notebook, we use muon kinematic data from CMS Open Data to reconstruct the three momentum components:

- $p_x$
- $p_y$
- $p_z$

We then calculate the total momentum and energy to construct the muon's four-momentum:

\[
(E, p_x, p_y, p_z)
\]

We will use a real opposite-charge muon pair from Event 78 as an example.

In [55]:
import uproot
import numpy as np

file = uproot.open(
    "../data/37E38CDA-3DD0-F844-A520-4FDF132B3A43.root"
)

events = file["Events"]

event_index = 78

muon_data = events.arrays(
    [
        "Muon_pt",
        "Muon_eta",
        "Muon_phi",
        "Muon_mass",
        "Muon_charge"
    ],
    entry_start=event_index,
    entry_stop=event_index + 1
)

print(muon_data)

muon0_pt = muon_data["Muon_pt"][0][0]
muon0_phi = muon_data["Muon_phi"][0][0]
muon0_eta = muon_data["Muon_eta"][0][0]
muon0_mass = muon_data["Muon_mass"][0][0]
muon0_charge = muon_data["Muon_charge"][0][0]

print("Muon 0 pt:", muon0_pt)
print("Muon 0 eta:", muon0_eta)
print("Muon 0 phi:", muon0_phi)
print("Muon 0 mass:", muon0_mass)
print("Muon 0 charge:", muon0_charge)

px0 = muon0_pt*np.cos(muon0_phi)
py0= muon0_pt*np.sin(muon0_phi)
pz0 = muon0_pt*np.sinh(muon0_eta)

print("px = ", px0)
print("py = ", py0)
print("pz = ", pz0)

p0 = np.sqrt(px0**2 + py0**2 + pz0**2)
print("Total momentum = ", p0)

E0 = np.sqrt(muon0_mass**2 + p0**2)
print("Energy = ", E0)

muon1_pt = muon_data["Muon_pt"][0][1]
muon1_phi = muon_data["Muon_phi"][0][1]
muon1_eta = muon_data["Muon_eta"][0][1]
muon1_mass = muon_data["Muon_mass"][0][1]
muon1_charge = muon_data["Muon_charge"][0][1]

print("Muon 1 pt:", muon1_pt)
print("Muon 1 eta:", muon1_eta)
print("Muon 1 phi:", muon1_phi)
print("Muon 1 mass:", muon1_mass)
print("Muon 1 charge:", muon1_charge)

px1 = muon1_pt * np.cos(muon1_phi)
py1 = muon1_pt * np.sin(muon1_phi)
pz1 = muon1_pt * np.sinh(muon1_eta)

print("Muon 1 px =", px1)
print("Muon 1 py =", py1)
print("Muon 1 pz =", pz1)

p1 = np.sqrt(px1**2 + py1**2 + pz1**2)

print("Muon 1 total momentum =", p1)

E1 = np.sqrt(muon1_mass**2 + p1**2)

print("Muon 1 energy =", E1)

E_total = E0 + E1

px_total = px0 + px1
py_total = py0 + py1
pz_total = pz0 + pz1

invariant_mass = np.sqrt(
    E_total**2
    -px_total**2
    -py_total**2
    -pz_total**2
)

print("Dimuon invariant mass =", invariant_mass, "GeV")




[{Muon_pt: [4.91, 3.38], Muon_eta: [0.351, 0.332], Muon_phi: [...], ...}]
Muon 0 pt: 4.9127684
Muon 0 eta: 0.3511963
Muon 0 phi: 2.4985352
Muon 0 mass: 0.10571289
Muon 0 charge: -1
px =  -3.931522
py =  2.9459174
pz =  1.7610325
Total momentum =  5.218863
Energy =  5.2199335
Muon 1 pt: 3.3847687
Muon 1 eta: 0.3317871
Muon 1 phi: -2.0874023
Muon 1 mass: 0.10571289
Muon 1 charge: 1
Muon 1 px = -1.6718453
Muon 1 py = -2.9430583
Muon 1 pz = 1.1437405
Muon 1 total momentum = 3.572786
Muon 1 energy = 3.5743496
Dimuon invariant mass = 6.124048 GeV


## From One Event to Multiple Events

We now extend the single-event analysis to multiple CMS events.

The goal is to identify events containing at least two muons with opposite charges.

In [ ]:
import awkward as ak

muon_batch = events.arrays(
    ["Muon_pt",
        "Muon_eta",
        "Muon_phi",
        "Muon_mass",
        "Muon_charge"
    ],
    entry_start=0,
    entry_stop=103100
)

print(muon_batch)

charges = muon_batch["Muon_charge"]
print(charges)

n_muons = ak.num(charges)

print(n_muons)

at_least_two_muons = n_muons >= 2
print(at_least_two_muons)

selected_charges = charges[at_least_two_muons]
print(selected_charges)

print("Total events:", len(charges))
print("Events with at least 2 muons:", len(selected_charges))

muon_pairs = ak.combinations(selected_charges, 2)
print(muon_pairs[:10])

charge_pairs = ak.unzip(muon_pairs)

charge1 = charge_pairs[0]
charge2 = charge_pairs[1]

opp_charges = charge1 * charge2 == -1
print(opp_charges[:10])

n_opp_pairs = ak.num(opp_charges)
total_opp_pairs = ak.sum(opp_charges)
print("Total opposite-charge muon pairs:", total_opp_pairs)

muon_data_selected = events.arrays(
    [
        "Muon_pt",
        "Muon_eta",
        "Muon_phi",
        "Muon_mass",
        "Muon_charge"
    ]
)

muon_data_selected = muon_data_selected[at_least_two_muons]

print(muon_data_selected[:3])

# Create one complete record for each muon
muons = ak.zip({
    "pt": muon_data_selected["Muon_pt"],
    "eta": muon_data_selected["Muon_eta"],
    "phi": muon_data_selected["Muon_phi"],
    "mass": muon_data_selected["Muon_mass"],
    "charge": muon_data_selected["Muon_charge"]
})

print(muons[:3])

muon_pairs_full = ak.combinations(
    muons,
    2,
    fields=["muon1", "muon2"]
)

print(muon_pairs_full[:3])

muon1 = muon_pairs_full["muon1"]
muon2 = muon_pairs_full["muon2"]

opp_charge_mask = (
    muon1["charge"] * muon2["charge"] == -1
)

print(opp_charge_mask[:10])

muon1_mask = muon1[opp_charge_mask]
muon2_mask = muon2[opp_charge_mask]

print(muon1_mask[:10])
print(muon2_mask[:10])

px1 = muon1_mask["pt"] * np.cos(muon1_mask["phi"])
py1 = muon1_mask["pt"] * np.sin(muon1_mask["phi"])
pz1 = muon1_mask["pt"] * np.sinh(muon1_mask["eta"])

p1 = np.sqrt(px1**2 + py1**2 + pz1**2)
muon1_mass = muon1_mask["mass"]

E1 = np.sqrt(p1**2 + muon1_mass**2)

px2 = muon2_mask["pt"] * np.cos(muon2_mask["phi"])
py2 = muon2_mask["pt"] * np.sin(muon2_mask["phi"])
pz2 = muon2_mask["pt"] * np.sinh(muon2_mask["eta"])

p2 = np.sqrt(px2**2 + py2**2 + pz2**2)
muon2_mass = muon2_mask["mass"]

E2 = np.sqrt(p2**2 + muon2_mass**2)

px_total = px1 + px2
py_total = py1 + py2
pz_total = pz1 + pz2

E_total = E1 + E2

mass_squared = (
    E_total**2
    - px_total**2
    - py_total**2
    - pz_total**2
)

mass_squared = np.maximum(mass_squared, 0) # to protect against tiny negative values caused by floating-point precision.

invariant_mass = np.sqrt(mass_squared)

print(invariant_mass[:10])

all_masses = ak.flatten(invariant_mass)

print(all_masses[:20])
print("Total invariant masses:", len(all_masses))



[{Muon_pt: [], Muon_eta: [], Muon_phi: [], Muon_mass: [], ...}, ..., {...}]
[[], [], [], [], [], [], [], [], [], ..., [1], [], [], [], [], [-1], [], [], []]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, ..., 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0]
[False, False, False, False, False, ..., False, False, False, False, False]
[[-1, -1], [-1, -1], [-1, 1], [-1, -1], ..., [-1, ...], [1, -1], [1, 1], [1, 1]]
Total events: 103100
Events with at least 2 muons: 2108
[[(-1, -1)], [(-1, -1)], [(-1, 1)], ..., [(-1, 1)], [(1, -1)], [(1, -1)]]
[[False], [False], [True], [False], ..., [False, ...], [True], [True], [True]]
Total opposite-charge muon pairs: 1187
[{Muon_pt: [6.68, 4.82], Muon_eta: [1.18, 1.02], Muon_phi: [...], ...}, ...]
[[{pt: 6.68, eta: 1.18, phi: 0.501, mass: 0.106, charge: -1}, {...}], ...]
[[{muon1: {pt: 6.68, eta: 1.18, ...}, muon2: {...}}], ..., [{muon1: ..., ...}]]
[[False], [False], [True], [False], ..., [False, ...], [True], [True], [True]]
[[], [], ..., [{pt: 4.11, eta: 1.36, phi: 2.71